# Assignment 02: Named Entity Recognition (NER) from a News Article

**Objective:**
Apply Named Entity Recognition (NER) techniques to identify and extract named entities from an English news article.

In [1]:
# Find a Data Source

**Source:** https://thehimalayantimes.com/opinion/2026-world-cup-football-tournament-the-greatest-sporting-extravaganza

In [2]:
# Load the data as a text object

In [3]:
with open("article.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(text)

The FIFA World Cup has finally come to an end with the lifting of the trophy by Spain after 16 years for the second time. It defeated the defending champion Argentina amidst thousands of spectators in New Jersey, including Donald Trump, the U.S. President. In a way, it has been a great relief for the Nepalis as they were spending sleepless nights, with most of the tournaments taking place at midnight. But the game's excitement and romance were so high that they simply overshadowed such marginal discomforts.
While the origins of football date back to cuju – an ancient game played in China during the Han Dynasty in the 3rd century – its early history in Nepal remains undocumented. For instance, the renowned Chinese traveller Huen Tsang detailed many facets of Nepalese life, such as art, architecture, and local skills, but made no mention of the sports played in the country. However, historical records confirm that wrestling was a popular sport in ancient Nepal. A 604 AD Licchavi inscript

In [4]:
# Named Entity Recognition

## Using SpaCy

In [5]:
import spacy

In [6]:
nlp = spacy.load("en_core_web_sm")

In [7]:
doc = nlp(text)

In [8]:
for ent in doc.ents:
    print(f"{ent.text:<45} | {ent.label_}")

World Cup                                     | EVENT
Spain                                         | GPE
16 years                                      | DATE
second                                        | ORDINAL
Argentina                                     | GPE
thousands                                     | CARDINAL
New Jersey                                    | GPE
Donald Trump                                  | PERSON
U.S.                                          | GPE
Nepalis                                       | PERSON
midnight                                      | TIME
China                                         | GPE
the Han Dynasty                               | ORG
the 3rd century                               | DATE
Nepal                                         | GPE
Chinese                                       | NORP
Huen Tsang                                    | PERSON
Nepalese                                      | NORP
Nepal                                  

In [24]:
# Visualizaion

In [25]:
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

In [9]:
# Exporting the data in a csv using pandas

In [10]:
import pandas as pd

In [11]:
df = pd.DataFrame(
    [{"Entity": ent.text, "Label": ent.label_} for ent in doc.ents]
)

df.to_csv("named_entities_spacy.csv", index=False)

**Overall Observations**

The article focuses on football, particularly the FIFA World Cup, Nepal, and related historical and political context.
Consequently, the most frequent entity categories are:

- GPE (Geo-Political Entities) – Most common
Countries and places such as Nepal, Argentina, Spain, Brazil, England, Iran, etc.

- PERSON
Football players, officials, politicians, and historical figures.
Examples: Messi, Maradona, Donald Trump, Gianni Infantino.

- ORG
Organizations like FIFA and sports associations.

- DATE
Numerous years and historical periods used to describe events.

- CARDINAL / ORDINAL
Numerical information such as rankings, counts, and positions.

Interesting Findings: spaCy **successfully recognized**

- Countries and cities (GPE)
- People (PERSON)
- Organizations (ORG)
- Sporting events (EVENT)
-Monetary values (MONEY)

There are also a few **incorrect classifications**, which are normal for pretrained NER models:
1. Siva Deb → classified as FAC instead of PERSON.
2. Cabo Verde → classified as PRODUCT instead of GPE.
3. Rs. 1.22 billion → classified as LAW instead of MONEY.
4. Round → classified as PRODUCT instead of a competition stage.

These errors highlight an important limitation of pretrained NER models: while they perform well on common entities, they can misclassify uncommon names, regional terms, or context-specific phrases.

For applications requiring high accuracy—such as legal, medical, or domain-specific text—fine-tuning the model or using a custom-trained NER model is often necessary.

In [12]:
# Get the frequency of label occurance 

In [13]:
df["Label"].value_counts()

Label
GPE         41
PERSON      20
DATE        16
CARDINAL    12
ORG         12
NORP        12
MONEY        8
EVENT        5
ORDINAL      4
LOC          2
PRODUCT      2
TIME         1
FAC          1
LAW          1
Name: count, dtype: int64

In [14]:
# Interpretation

As told earlier in summary, GPE has the highest occurance in the article with occurance count of 41 followed by PERSON with 20.

## Using NLTK

In [15]:
import nltk
from nltk.tree import Tree

In [16]:
nltk.download("punkt")              # Sentence and word tokenizer
nltk.download("averaged_perceptron_tagger")  # POS tagger
nltk.download("maxent_ne_chunker")  # Named Entity Recognition model
nltk.download("words")              # English word corpus used by the NER model

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sleepdeprived/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/sleepdeprived/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/sleepdeprived/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     /Users/sleepdeprived/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [17]:
# Split the text into individual words
tokens = nltk.word_tokenize(text)

# Assign Part-of-Speech (POS) tags to each token
pos_tags = nltk.pos_tag(tokens)

# Perform Named Entity Recognition (returns a parse tree)
ner_tree = nltk.ne_chunk(pos_tags)

In [18]:
# Store extracted entities
data = []

# Iterate through the parse tree
for subtree in ner_tree:

    # If the node is a Tree, it is a named entity
    if isinstance(subtree, Tree):

        # Combine multi-word entities into a single string
        entity = " ".join(word for word, pos in subtree.leaves())

        # Get the entity label (PERSON, GPE, ORGANIZATION, etc.)
        label = subtree.label()

        # Append to our list
        data.append([entity, label])

# Create a DataFrame
df = pd.DataFrame(data, columns=["Entity", "Label"])

# Save to CSV
df.to_csv("named_entities_nltk.csv", index=False)

## Using Hugging Face Transformer

In [19]:
from transformers import pipeline
import torch

In [20]:
# Load a pretrained NER model from Hugging Face
ner = pipeline(
    "ner",
    model="tner/roberta-large-ontonotes5",
    aggregation_strategy="simple"
)

# Run Named Entity Recognition
entities = ner(text)

# Print detected entities
for entity in entities:
    print(entity)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

{'entity_group': 'EVENT', 'score': np.float32(0.9994955), 'word': 'The FIFA World Cup', 'start': 0, 'end': 18}
{'entity_group': 'GPE', 'score': np.float32(0.99848455), 'word': ' Spain', 'start': 80, 'end': 85}
{'entity_group': 'DATE', 'score': np.float32(0.9992485), 'word': ' 16 years', 'start': 92, 'end': 100}
{'entity_group': 'ORDINAL', 'score': np.float32(0.9947305), 'word': ' second', 'start': 109, 'end': 115}
{'entity_group': 'GPE', 'score': np.float32(0.82777154), 'word': '.', 'start': 120, 'end': 121}
{'entity_group': 'GPE', 'score': np.float32(0.99874663), 'word': ' Argentina', 'start': 157, 'end': 166}
{'entity_group': 'CARDINAL', 'score': np.float32(0.81716883), 'word': ' thousands', 'start': 174, 'end': 183}
{'entity_group': 'GPE', 'score': np.float32(0.9997395), 'word': ' New Jersey', 'start': 201, 'end': 211}
{'entity_group': 'PERSON', 'score': np.float32(0.9951333), 'word': ' Donald Trump', 'start': 223, 'end': 235}
{'entity_group': 'GPE', 'score': np.float32(0.999181), '

In [21]:
# Create a DataFrame from the returned list of dictionaries
df = pd.DataFrame(entities)

# Save the DataFrame as a CSV file
df.to_csv("named_entities_huggingface.csv", index=False)

## Comparision

In [22]:
"""
Builds the human (gold-standard) benchmark for NER evaluation.

The list below is a manual, human-read annotation of /mnt/user-data/uploads/article.txt
(entity text + label), listed IN THE ORDER the entities occur in the article.
This script locates the exact character offsets of each entity by scanning the
article text sequentially (a moving cursor), so that repeated mentions of the
same string (e.g. "Nepal", "FIFA", "Argentina") are matched to the correct
occurrence rather than always the first one.

Label scheme: OntoNotes-style (same tag set spaCy's en_core_web_sm uses), since
this is the richest of the three schemes produced in the assignment notebook and
every other system's labels can be mapped onto it (see evaluate.py::LABEL_MAP).
"""

with open("article.txt", encoding="utf-8") as f:
    TEXT = f.read()

GOLD = [
    ("FIFA World Cup", "EVENT"),
    ("Spain", "GPE"),
    ("16 years", "DATE"),
    ("second", "ORDINAL"),
    ("Argentina", "GPE"),
    ("thousands", "CARDINAL"),
    ("New Jersey", "GPE"),
    ("Donald Trump", "PERSON"),
    ("U.S.", "GPE"),
    ("Nepalis", "NORP"),
    ("midnight", "TIME"),

    ("cuju", "PRODUCT"),
    ("China", "GPE"),
    ("the Han Dynasty", "ORG"),
    ("the 3rd century", "DATE"),
    ("Nepal", "GPE"),
    ("Chinese", "NORP"),
    ("Huen Tsang", "PERSON"),
    ("Nepalese", "NORP"),
    ("Nepal", "GPE"),
    ("604 AD", "DATE"),
    ("Licchavi", "NORP"),
    ("Siva Deb", "PERSON"),
    ("Amshu Verma", "PERSON"),
    ("68 manika", "MONEY"),

    ("Britain", "GPE"),
    ("the 15th century", "DATE"),
    ("the industrial revolution", "EVENT"),

    ("International Federation of Association Football", "ORG"),
    ("FIFA", "ORG"),
    ("1904", "DATE"),
    ("Uruguay", "GPE"),
    ("first", "ORDINAL"),
    ("World Cup", "EVENT"),
    ("1930", "DATE"),
    ("Latin American", "NORP"),
    ("European", "NORP"),
    ("Asian", "NORP"),
    ("African", "NORP"),
    ("This year", "DATE"),
    ("Asian", "NORP"),
    ("African", "NORP"),
    ("Europe", "GPE"),
    ("Latin America", "GPE"),

    ("Nepal", "GPE"),
    ("1951", "DATE"),
    ("Nepal", "GPE"),
    ("1921", "DATE"),
    ("the Rana regime", "ORG"),
    ("FIFA", "ORG"),
    ("1972", "DATE"),
    ("This year", "DATE"),
    ("Nepali", "NORP"),
    ("World Cup", "EVENT"),
    ("ANFA", "ORG"),
    ("FIFA", "ORG"),
    ("the Nepal Sports Council", "ORG"),
    ("ANFA", "ORG"),

    ("Nepal", "GPE"),
    ("177th", "ORDINAL"),
    ("Group H", "EVENT"),
    ("second", "ORDINAL"),
    ("the United Arab Emirates", "GPE"),
    ("Bahrain", "GPE"),
    ("Yemen", "GPE"),
    ("Laos", "GPE"),
    ("Nepali", "NORP"),
    ("88th", "ORDINAL"),

    ("Brazil", "GPE"),
    ("Argentina", "GPE"),
    ("Nepal", "GPE"),
    ("Argentina", "GPE"),
    ("Brazil", "GPE"),
    ("Argentina", "GPE"),
    ("Messi", "PERSON"),
    ("Maradona", "PERSON"),

    ("Maradona", "PERSON"),
    ("Nepal", "GPE"),
    ("England", "GPE"),
    ("Argentina", "GPE"),
    ("1982", "DATE"),
    ("Falklands War", "EVENT"),
    ("Maradona", "PERSON"),
    ("England", "GPE"),
    ("the Gurkhas", "NORP"),
    ("England", "GPE"),
    ("Argentina", "GPE"),
    ("2-1", "CARDINAL"),
    ("hand of God", "EVENT"),
    ("Maradona", "PERSON"),

    ("This year", "DATE"),
    ("World Cup", "EVENT"),
    ("Cabol Verde", "GPE"),
    ("the Round of 32", "EVENT"),
    ("Spain", "GPE"),
    ("3-2", "CARDINAL"),
    ("Argentina", "GPE"),
    ("40-year-old", "DATE"),
    ("Vozinha", "PERSON"),
    ("seven", "CARDINAL"),
    ("19-year-old", "DATE"),
    ("Spanish", "NORP"),
    ("Lamine Yamal", "PERSON"),
    ("French", "NORP"),
    ("Mbappe", "PERSON"),
    ("the golden boot award", "WORK_OF_ART"),
    ("10", "CARDINAL"),

    ("Iran", "GPE"),
    ("US", "GPE"),
    ("the Middle East", "LOC"),
    ("Iran", "GPE"),
    ("three", "CARDINAL"),
    ("New Zealand", "GPE"),
    ("Belgium", "GPE"),
    ("Egypt", "GPE"),

    ("the United States", "GPE"),
    ("Folarin Balegun", "PERSON"),
    ("one", "CARDINAL"),
    ("U.S.", "GPE"),
    ("Trump", "PERSON"),
    ("FIFA", "ORG"),
    ("Gianni Infantino", "PERSON"),

    ("$15 billion", "MONEY"),
    ("$4 billion", "MONEY"),
    ("$11 billion", "MONEY"),
    ("$655 million", "MONEY"),
    ("Spain", "GPE"),
    ("$50 million", "MONEY"),
    ("Argentina", "GPE"),
    ("$ 33 million", "MONEY"),
    ("Three billion dollars", "MONEY"),
    ("FIFA", "ORG"),
    ("Canada", "GPE"),
    ("US", "GPE"),
    ("Mexico", "GPE"),
    ("$8.3 billion", "MONEY"),
    ("Global Football Development", "ORG"),
    ("Nepal", "GPE"),
    ("Rs 1.22 billion", "MONEY"),

    ("Himalayan Sports", "ORG"),
    ("Nepal", "GPE"),
    ("48", "CARDINAL"),
    ("Balen Shah", "PERSON"),
    ("500,000", "CARDINAL"),
    ("Cabo Verde", "GPE"),
    ("Nepal", "GPE"),
]

rows = []
cursor = 0
for text, label in GOLD:
    idx = TEXT.find(text, cursor)
    if idx == -1:
        # fall back to searching from the start
        idx = TEXT.find(text)
        if idx == -1:
            raise ValueError(f"Could not locate {text!r} in article text")
    start, end = idx, idx + len(text)
    rows.append({"Entity": text, "Label": label, "Start": start, "End": end})
    cursor = end

df = pd.DataFrame(rows)
df.to_csv("benchmark.csv", index=False)
print(f"Wrote {len(df)} gold entities to benchmark.csv")
print(df["Label"].value_counts())

Wrote 146 gold entities to benchmark.csv
Label
GPE            50
PERSON         16
DATE           15
NORP           15
ORG            13
MONEY          10
EVENT           9
CARDINAL        9
ORDINAL         5
TIME            1
PRODUCT         1
WORK_OF_ART     1
LOC             1
Name: count, dtype: int64


In [23]:
# Comparing NER outputs against the human benchmark

import pandas as pd
from collections import Counter

gold = pd.read_csv("benchmark.csv")

spacy_df = pd.read_csv("named_entities_spacy.csv")
nltk_df  = pd.read_csv("named_entities_nltk.csv")
hf_df    = pd.read_csv("named_entities_huggingface.csv")

# standardize each to two columns: Entity, Label
spacy_df = spacy_df[["Entity", "Label"]]
nltk_df  = nltk_df[["Entity", "Label"]]
hf_df    = hf_df.rename(columns={"word": "Entity", "entity_group": "Label"})[["Entity", "Label"]]
gold     = gold[["Entity", "Label"]]

systems = {
    "spaCy": spacy_df,
    "NLTK": nltk_df,
    "HuggingFace": hf_df,
}

def normalize(df):
    return list(zip(df["Entity"].astype(str).str.strip().str.lower(),
                     df["Label"].astype(str).str.strip().str.upper()))

def evaluate(gold_df, pred_df):
    gold_pairs = normalize(gold_df)
    pred_pairs = normalize(pred_df)

    gold_text_only = [t for t, l in gold_pairs]
    pred_text_only = [t for t, l in pred_pairs]

    # strict match: entity text AND label both match
    strict_hits = sum((Counter(gold_pairs) & Counter(pred_pairs)).values())

    # text-only match: entity text matches, label ignored
    text_hits = sum((Counter(gold_text_only) & Counter(pred_text_only)).values())

    n_gold, n_pred = len(gold_pairs), len(pred_pairs)

    def prf(hits, n_pred, n_gold):
        p = hits / n_pred if n_pred else 0
        r = hits / n_gold if n_gold else 0
        f1 = 2 * p * r / (p + r) if (p + r) else 0
        return p, r, f1

    strict_p, strict_r, strict_f1 = prf(strict_hits, n_pred, n_gold)
    text_p, text_r, text_f1 = prf(text_hits, n_pred, n_gold)

    # label accuracy among entities whose text was found in both
    label_accuracy = strict_hits / text_hits if text_hits else 0

    return {
        "n_gold": n_gold, "n_pred": n_pred,
        "strict_precision": round(strict_p, 3), "strict_recall": round(strict_r, 3), "strict_f1": round(strict_f1, 3),
        "entity_precision": round(text_p, 3), "entity_recall": round(text_r, 3), "entity_f1": round(text_f1, 3),
        "label_accuracy": round(label_accuracy, 3),
    }

# Run comparison 
results = {name: evaluate(gold, df) for name, df in systems.items()}
comparison = pd.DataFrame(results).T
comparison

,n_gold,n_pred,strict_precision,strict_recall,strict_f1,entity_precision,entity_recall,entity_f1,label_accuracy
spaCy,146.0,137.0,0.774,0.726,0.749,0.869,0.815,0.841,0.891
NLTK,146.0,95.0,0.453,0.295,0.357,0.842,0.548,0.664,0.537
HuggingFace,146.0,73.0,0.658,0.329,0.438,0.699,0.349,0.466,0.941
